# Adapting DeepGaze MSDB to a new dataset

DeepGaze MSDB carries a small set of **per-dataset parameters** (13 scalars: multi-scale weights, a gaussian blur sigma, a center-bias weight and a priority scaling) on top of a frozen CLIP+DINOv2 backbone and a shared saliency network. To adapt the model to a *new* dataset we add one new parameter slot and train only those 13 scalars.

The new slot is initialised from the average of the model's original datasets, so it starts from the model's generalization behaviour (`dataset=None`) and only needs a little data to specialise. Adapting does **not** change the predictions for the built-in datasets.

This notebook walks through adapting the released model to the public **Toronto** dataset. Replace the data-loading cell with your own `images/` folder and a `fixations.csv` (columns `image,x,y`) to adapt to your data.

In [ ]:
import numpy as np
import torch

import deepgaze_pytorch
from deepgaze_pytorch import DeepGazeMSDB, MSDBDataset
from deepgaze_pytorch.custom_data import load_fixations_csv, fit_centerbias
from deepgaze_pytorch.msdb_finetuning import finetune_new_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load your data

`load_fixations_csv` turns an image folder + a fixation CSV into pysaliency `stimuli, fixations`. If you already have pysaliency objects, skip this and use them directly.

Here we build such a CSV from the Toronto dataset shipped with pysaliency, purely to have a concrete public example.

In [ ]:
import csv, os
import pysaliency

# Get Toronto from pysaliency (downloads on first use) and write an images-folder + CSV,
# so this example uses exactly the same public entry point as your own data would.
toronto_stimuli, toronto_fixations = pysaliency.external_datasets.get_toronto(location='pysaliency_datasets')
image_dir = os.path.join('pysaliency_datasets', 'toronto', 'stimuli')  # get_toronto writes the images here
os.makedirs('adaptation_example', exist_ok=True)
csv_path = os.path.join('adaptation_example', 'toronto_fixations.csv')
names = [os.path.basename(f) for f in toronto_stimuli.filenames]
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['image', 'x', 'y'])
    for x, y, n in zip(toronto_fixations.x, toronto_fixations.y, toronto_fixations.n):
        w.writerow([names[int(n)], float(x), float(y)])

# --- this is the part you would run on your own data: ---
stimuli, fixations = load_fixations_csv(image_dir, csv_path)
print(len(stimuli), 'images,', len(fixations), 'fixations')

PIXEL_PER_DVA = 21.75   # pixels per degree of visual angle of your presentation (Toronto ~21.75)

## 2. Fit a center bias

DeepGaze models take a center-bias (a baseline log-density of where fixations land, independent of the image) as input. `fit_centerbias` fits one over your fixations; pass your own model instead if you have one.

In [ ]:
centerbias = fit_centerbias(stimuli, fixations)

## 3. Hold out some images for validation

We split by image so validation measures generalization to unseen images.

In [ ]:
from pysaliency.filter_datasets import train_split, validation_split
train_st, train_fx = train_split(stimuli, fixations, crossval_folds=10, fold_no=0)
val_st, val_fx = validation_split(stimuli, fixations, crossval_folds=10, fold_no=0)
print('train', len(train_st), 'images; val', len(val_st), 'images')

## 4. Load the released model and add a dataset slot

`add_dataset()` appends the new slot and returns its index. As a sanity check, at initialisation the new slot reproduces the averaged `dataset=None` prediction exactly.

In [ ]:
model = DeepGazeMSDB(pretrained=True).to(DEVICE).eval()

def predict(stimulus, dataset):
    cb = centerbias.log_density(stimulus)
    img = torch.tensor(np.array([np.asarray(stimulus).transpose(2, 0, 1)]), dtype=torch.float32, device=DEVICE)
    cbt = torch.tensor(np.array([cb]), dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return model(img, cbt, pixel_per_dva=PIXEL_PER_DVA, dataset=dataset)[0].cpu().numpy()

probe = val_st.stimuli[0]
averaged = predict(probe, None)
dataset_index = model.add_dataset()
print('new dataset index:', dataset_index)
print('init reproduces dataset=None, max abs diff:',
      np.abs(predict(probe, dataset_index) - averaged).max())

## 5. Adapt (train the 13 parameters)

`finetune_new_dataset` trains only the new slot. We pass the `dataset_index` we just created so it does not add a second slot. The head-only adapted weights are written to `train_directory/final.pth`.

In [ ]:
model = finetune_new_dataset(
    model, train_st, train_fx, centerbias, PIXEL_PER_DVA,
    train_directory='adaptation_example/run',
    val_stimuli=val_st, val_fixations=val_fx,
    dataset_index=dataset_index,
)

## 6. Compare zero-shot vs adapted

We score information gain (bit/fixation over the center bias) on the held-out images, for the averaged (`dataset=None`) model and for the adapted slot.

In [ ]:
def mean_ig(dataset):
    class _M(pysaliency.Model):
        def __init__(s): super().__init__(caching=False)
        def _log_density(s, stim):
            return predict(pysaliency.datasets.as_stimulus(stim), dataset)
    m = _M()
    lls = m.log_likelihoods(val_st, val_fx)
    base = centerbias.log_likelihoods(val_st, val_fx)
    return float(np.mean(lls - base))

print('zero-shot (dataset=None) IG:', round(mean_ig(None), 4))
print('adapted           IG:', round(mean_ig(dataset_index), 4))

## Notes and next steps

- On Toronto the gain from adaptation is small — the averaged parameters already generalize well. Other datasets may benefit more.
- The adapted model still serves the built-in datasets (`dataset=MSDBDataset.MIT1003`, ...) unchanged, and you can adapt to several datasets in turn.
- To reload `adaptation_example/run/final.pth` later: build a `DeepGazeMSDB`, call `model.add_dataset()`, then `model.load_state_dict(torch.load(...), strict=False)`.
- **Ideas to extend this notebook:** visualise the learned per-dataset parameters (scale weights, sigma, center-bias weight) before vs after adaptation, or plot a few predicted density maps against the ground-truth fixations.